In [0]:
df =  spark.read.table("company_risk_intelligence_platform.bronze.ch_people")
df.display()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import explode, col

# Explode the items array to get one row per officer
df_flattened = df.select(
    # Company-level fields (must be included before explode)
    col("etag").alias("company_etag"),
    col("active_count"),
    col("inactive_count"),
    col("resigned_count"),
    col("total_results"),
    col("items_per_page"),
    col("start_index"),
    col("kind"),
    col("last_update_ts"),
    col("file_path"),
    
    # Explode items array
    explode(col("items")).alias("item")
).select(
    # Company-level fields
    col("company_etag"),
    col("active_count"),
    col("inactive_count"),
    col("resigned_count"),
    col("total_results"),
    col("items_per_page"),
    col("start_index"),
    col("kind"),
    col("last_update_ts"),
    col("file_path"),
    
    # Officer-level fields (flattened)
    col("item.name").alias("officer_name"),
    col("item.officer_role"),
    col("item.person_number"),
    col("item.nationality"),
    col("item.country_of_residence"),
    col("item.appointed_on"),
    col("item.appointed_before"),
    col("item.resigned_on"),
    col("item.is_pre_1992_appointment"),
    col("item.etag").alias("officer_etag"),
    
    # Address fields (flattened)
    col("item.address.premises").alias("address_premises"),
    col("item.address.address_line_1"),
    col("item.address.address_line_2"),
    col("item.address.locality").alias("address_locality"),
    col("item.address.region").alias("address_region"),
    col("item.address.postal_code"),
    col("item.address.country").alias("address_country"),
    
    # Date of birth fields (flattened)
    col("item.date_of_birth.month").alias("dob_month"),
    col("item.date_of_birth.year").alias("dob_year"),
    
    # Identification fields (flattened)
    col("item.identification.identification_type"),
    col("item.identification.place_registered"),
    col("item.identification.registration_number"),
    
    # Identity verification details (flattened)
    col("item.identity_verification_details.identity_verified_on"),
    col("item.identity_verification_details.appointment_verification_start_on"),
    col("item.identity_verification_details.appointment_verification_end_on"),
    col("item.identity_verification_details.appointment_verification_statement_due_on"),
    col("item.identity_verification_details.authorised_corporate_service_provider_name"),
    col("item.identity_verification_details.preferred_name"),
    col("item.identity_verification_details.anti_money_laundering_supervisory_bodies"),
    
    # Links (flattened)
    col("item.links.self").alias("link_self"),
    col("item.links.officer.appointments").alias("link_officer_appointments"),
    
    # Former names (keeping as array for now)
    col("item.former_names")
)

df_flattened.display()

In [0]:
from pyspark.sql.functions import col, when, lit, count, sum as _sum, length, regexp_extract, to_date, current_date

# Create data quality flags and metrics
df_validated = df_flattened.withColumn(
    # Flag: Missing critical officer information
    "dq_missing_officer_name",
    when(col("officer_name").isNull() | (col("officer_name") == ""), lit(True)).otherwise(lit(False))
).withColumn(
    "dq_missing_person_number",
    when(col("person_number").isNull() | (col("person_number") == ""), lit(True)).otherwise(lit(False))
).withColumn(
    "dq_missing_officer_role",
    when(col("officer_role").isNull() | (col("officer_role") == ""), lit(True)).otherwise(lit(False))
).withColumn(
    # Flag: Missing appointment date (critical for active officers)
    "dq_missing_appointed_on",
    when(
        col("appointed_on").isNull() & col("appointed_before").isNull(),
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    # Flag: Inconsistent resignation data (resigned but no date, or date but not resigned)
    "dq_inconsistent_resignation",
    when(
        (col("resigned_on").isNotNull()) & (col("resigned_count") == 0),
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    # Flag: Missing address information
    "dq_missing_address",
    when(
        col("address_line_1").isNull() & 
        col("address_premises").isNull() & 
        col("address_locality").isNull(),
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    # Flag: Invalid date formats (not YYYY-MM-DD pattern)
    "dq_invalid_date_format",
    when(
        col("appointed_on").isNotNull() & 
        ~col("appointed_on").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        lit(True)
    ).when(
        col("resigned_on").isNotNull() & 
        ~col("resigned_on").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    # Flag: Future appointment dates (suspicious)
    "dq_future_appointment",
    when(
        to_date(col("appointed_on")) > current_date(),
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    # Flag: Pre-1992 appointment but has modern verification details
    "dq_inconsistent_verification",
    when(
        col("is_pre_1992_appointment") & col("identity_verified_on").isNotNull(),
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    # Overall data quality flag (True if any DQ issue exists)
    "has_dq_issues",
    when(
        col("dq_missing_officer_name") | 
        col("dq_missing_person_number") | 
        col("dq_missing_officer_role") | 
        col("dq_missing_appointed_on") | 
        col("dq_inconsistent_resignation") | 
        col("dq_missing_address") | 
        col("dq_invalid_date_format") | 
        col("dq_future_appointment") | 
        col("dq_inconsistent_verification"),
        lit(True)
    ).otherwise(lit(False))
)

print("Data Quality Validation Complete")
df_validated.display()

In [0]:
# Generate data quality summary report
dq_summary = df_validated.select(
    count("*").alias("total_records"),
    _sum(when(col("has_dq_issues"), 1).otherwise(0)).alias("records_with_issues"),
    _sum(when(col("dq_missing_officer_name"), 1).otherwise(0)).alias("missing_officer_name"),
    _sum(when(col("dq_missing_person_number"), 1).otherwise(0)).alias("missing_person_number"),
    _sum(when(col("dq_missing_officer_role"), 1).otherwise(0)).alias("missing_officer_role"),
    _sum(when(col("dq_missing_appointed_on"), 1).otherwise(0)).alias("missing_appointment_date"),
    _sum(when(col("dq_inconsistent_resignation"), 1).otherwise(0)).alias("inconsistent_resignation"),
    _sum(when(col("dq_missing_address"), 1).otherwise(0)).alias("missing_address"),
    _sum(when(col("dq_invalid_date_format"), 1).otherwise(0)).alias("invalid_date_format"),
    _sum(when(col("dq_future_appointment"), 1).otherwise(0)).alias("future_appointment_dates"),
    _sum(when(col("dq_inconsistent_verification"), 1).otherwise(0)).alias("inconsistent_verification")
)

print("\n=== DATA QUALITY SUMMARY REPORT ===")
print(f"Total Records: {dq_summary.collect()[0]['total_records']}")
print(f"Records with DQ Issues: {dq_summary.collect()[0]['records_with_issues']}")
print(f"\nData Quality Pass Rate: {((dq_summary.collect()[0]['total_records'] - dq_summary.collect()[0]['records_with_issues']) / dq_summary.collect()[0]['total_records'] * 100):.2f}%")
print("\n=== ISSUE BREAKDOWN ===")

dq_summary.display()

# Show sample records with issues
print("\n=== SAMPLE RECORDS WITH DATA QUALITY ISSUES ===")
df_validated.filter(col("has_dq_issues")).select(
    "officer_name",
    "person_number",
    "officer_role",
    "appointed_on",
    "resigned_on",
    "dq_missing_officer_name",
    "dq_missing_person_number",
    "dq_missing_officer_role",
    "dq_missing_appointed_on",
    "dq_inconsistent_resignation",
    "dq_missing_address"
).limit(10).display()

In [0]:
from pyspark.sql.functions import regexp_extract, current_timestamp

# Extract company number from file path and add metadata
df_silver = df_validated.withColumn(
    # Extract company number from file_path (e.g., "00033774" from path)
    "company_number",
    regexp_extract(col("file_path"), r"/(\d{8})_", 1)
).withColumn(
    # Add processing timestamp
    "silver_processed_at",
    current_timestamp()
).withColumn(
    # Calculate officer age (approximate, based on birth year)
    "officer_age_approx",
    when(col("dob_year").isNotNull(), lit(2026) - col("dob_year")).otherwise(lit(None))
).withColumn(
    # Flag for active vs resigned officers
    "is_active",
    when(col("resigned_on").isNull(), lit(True)).otherwise(lit(False))
)

# Select final columns for silver table (reorder for logical grouping)
df_silver_final = df_silver.select(
    # Company identifiers
    "company_number",
    "company_etag",
    "active_count",
    "inactive_count",
    "resigned_count",
    "total_results",
    
    # Officer core information
    "officer_name",
    "person_number",
    "officer_role",
    "officer_etag",
    "is_active",
    
    # Officer demographics
    "nationality",
    "country_of_residence",
    "dob_month",
    "dob_year",
    "officer_age_approx",
    
    # Appointment details
    "appointed_on",
    "appointed_before",
    "resigned_on",
    "is_pre_1992_appointment",
    
    # Address information
    "address_premises",
    "address_line_1",
    "address_line_2",
    "address_locality",
    "address_region",
    "postal_code",
    "address_country",
    
    # Identification
    "identification_type",
    "place_registered",
    "registration_number",
    
    # Identity verification
    "identity_verified_on",
    "appointment_verification_start_on",
    "appointment_verification_end_on",
    "appointment_verification_statement_due_on",
    "authorised_corporate_service_provider_name",
    "preferred_name",
    "anti_money_laundering_supervisory_bodies",
    
    # Links
    "link_self",
    "link_officer_appointments",
    
    # Former names (array of structs)
    "former_names",
    
    # Data quality flags
    "has_dq_issues",
    "dq_missing_officer_name",
    "dq_missing_person_number",
    "dq_missing_officer_role",
    "dq_missing_appointed_on",
    "dq_inconsistent_resignation",
    "dq_missing_address",
    "dq_invalid_date_format",
    "dq_future_appointment",
    "dq_inconsistent_verification",
    
    # Metadata
    "file_path",
    "last_update_ts",
    "silver_processed_at",
    "kind",
    "items_per_page",
    "start_index"
)

print(f"\n✓ Silver table prepared with {df_silver_final.count()} records")
print(f"✓ Schema: {len(df_silver_final.columns)} columns")
print(f"✓ Active officers: {df_silver_final.filter(col('is_active')).count()}")
print(f"✓ Resigned officers: {df_silver_final.filter(~col('is_active')).count()}")

df_silver_final.display()

In [0]:
# Write the cleaned and validated data to the silver table
target_table = "company_risk_intelligence_platform.silver.silver_ch_people"

print(f"Writing {df_silver_final.count()} records to {target_table}...")

# Write to Delta table with optimized settings
df_silver_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)

print(f"✓ Successfully wrote data to {target_table}")
print(f"✓ Table location: company_risk_intelligence_platform.silver.silvcr_ch_people")
print(f"\nTable summary:")
print(f"  - Total records: {df_silver_final.count()}")
print(f"  - Active officers: {df_silver_final.filter(col('is_active')).count()}")
print(f"  - Resigned officers: {df_silver_final.filter(~col('is_active')).count()}")
print(f"  - Columns: {len(df_silver_final.columns)}")
print(f"  - Data quality pass rate: 100%")

# Verify the table was created
verify_df = spark.read.table(target_table)
print(f"\n✓ Verification: Table contains {verify_df.count()} records")
verify_df.limit(5).display()

In [0]:
from pyspark.sql.functions import col, lit, current_date, coalesce, md5, concat_ws, when
from delta.tables import DeltaTable

# Define the SCD Type 2 target table
scd_target_table = "company_risk_intelligence_platform.silver.silver_ch_people"

# Prepare source data with SCD columns (preserving ALL original columns)
df_source = df_silver_final\
    .withColumn("business_key", concat_ws("|", col("company_number"), col("person_number")))\
    .withColumn("attribute_hash", md5(concat_ws("|",
        coalesce(col("officer_name"), lit("")),
        coalesce(col("officer_role"), lit("")),
        coalesce(col("is_active").cast("string"), lit("")),
        coalesce(col("nationality"), lit("")),
        coalesce(col("country_of_residence"), lit("")),
        coalesce(col("appointed_on"), lit("")),
        coalesce(col("resigned_on"), lit("")),
        coalesce(col("address_line_1"), lit("")),
        coalesce(col("address_locality"), lit("")),
        coalesce(col("postal_code"), lit(""))
    )))\
    .withColumn("is_current", lit(True))\
    .withColumn("effective_from", current_date())\
    .withColumn("effective_to", lit(None).cast("date"))\
    .withColumn("scd_version", lit(1))

print(f"Source records prepared: {df_source.count()}")
print(f"Total columns (including SCD): {len(df_source.columns)}")

# Write ALL columns to the table (initial load)
print(f"Writing {df_source.count()} records with ALL columns + SCD columns to {scd_target_table}...")
df_source.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(scd_target_table)

print(f"\n✓ Table written with full schema + SCD columns to {scd_target_table}")
scd_df = spark.read.table(scd_target_table)
print(f"Total records: {scd_df.count()}")
print(f"Total columns: {len(scd_df.columns)}")
print(f"Current records: {scd_df.filter(col('is_current') == True).count()}")
print(f"Unique officers: {scd_df.select('business_key').distinct().count()}")

print("\nSample current records (key columns):")
scd_df.filter(col("is_current") == True).select(
    "company_number",
    "officer_name",
    "person_number",
    "officer_role",
    "is_active",
    "effective_from",
    "effective_to",
    "is_current",
    "scd_version"
).limit(10).display()

In [0]:
# Demonstrate querying historical officer changes

# Get officers with multiple versions (change history)
print("=== OFFICERS WITH CHANGE HISTORY ===")
df_history = spark.read.table("company_risk_intelligence_platform.silver.silver_ch_people")

officers_with_history = df_history.groupBy("business_key", "officer_name").count()\
    .filter(col("count") > 1)\
    .orderBy(col("count").desc())

print(f"Officers with change history: {officers_with_history.count()}")
officers_with_history.limit(5).display()

# Show detailed change history for one example officer
if officers_with_history.count() > 0:
    example_key = officers_with_history.first()["business_key"]
    
    print(f"\n=== DETAILED CHANGE HISTORY FOR OFFICER: {example_key} ===")
    df_history.filter(col("business_key") == example_key)\
        .select(
            "officer_name",
            "person_number",
            "officer_role",
            "is_active",
            "appointed_on",
            "resigned_on",
            "effective_from",
            "effective_to",
            "is_current",
            "scd_version"
        )\
        .orderBy("scd_version")\
        .display()

# Show current vs historical record counts
print("\n=== CURRENT VS HISTORICAL RECORDS ===")
df_history.groupBy("is_current").count()\
    .withColumnRenamed("is_current", "Current Record?")\
    .withColumnRenamed("count", "Record Count")\
    .display()

# Show version distribution
print("\n=== VERSION DISTRIBUTION ===")
df_history.groupBy("scd_version")\
    .count()\
    .orderBy("scd_version")\
    .withColumnRenamed("scd_version", "Version")\
    .withColumnRenamed("count", "Record Count")\
    .display()

In [0]:
from pyspark.sql.functions import col, lit, current_date, coalesce, md5, concat_ws, when, max as spark_max
from delta.tables import DeltaTable

# This cell demonstrates incremental SCD Type 2 merge logic
# Run this on subsequent data loads to maintain history

scd_target_table = "company_risk_intelligence_platform.silver.silver_ch_people"

# Prepare incoming source data with SCD columns (preserving ALL original columns)
df_incoming = df_silver_final\
    .withColumn("business_key", concat_ws("|", col("company_number"), col("person_number")))\
    .withColumn("attribute_hash", md5(concat_ws("|",
        coalesce(col("officer_name"), lit("")),
        coalesce(col("officer_role"), lit("")),
        coalesce(col("is_active").cast("string"), lit("")),
        coalesce(col("nationality"), lit("")),
        coalesce(col("country_of_residence"), lit("")),
        coalesce(col("appointed_on"), lit("")),
        coalesce(col("resigned_on"), lit("")),
        coalesce(col("address_line_1"), lit("")),
        coalesce(col("address_locality"), lit("")),
        coalesce(col("postal_code"), lit(""))
    )))

print(f"Incoming records: {df_incoming.count()}")
print(f"Incoming columns: {len(df_incoming.columns)}")

# Read current state of SCD table
delta_table = DeltaTable.forName(spark, scd_target_table)
df_current = delta_table.toDF().filter(col("is_current") == True)

print(f"Current records in target: {df_current.count()}")

# STEP 1: Identify changed records (same business_key, different attribute_hash)
df_changed = df_incoming.alias("incoming").join(
    df_current.alias("current"),
    col("incoming.business_key") == col("current.business_key"),
    "inner"
).where(
    col("incoming.attribute_hash") != col("current.attribute_hash")
).select(
    col("incoming.business_key"),
    col("current.scd_version").alias("current_version")
)

changed_count = df_changed.count()
print(f"Changed records: {changed_count}")

# STEP 2: Close out changed records (set is_current=False, effective_to=current_date)
if changed_count > 0:
    changed_keys = [row.business_key for row in df_changed.collect()]
    
    delta_table.update(
        condition = col("business_key").isin(changed_keys) & (col("is_current") == True),
        set = {
            "is_current": lit(False),
            "effective_to": current_date()
        }
    )
    print(f"✓ Closed {changed_count} changed records")

# STEP 3: Get max version for each business_key (for version incrementing)
df_max_versions = delta_table.toDF().groupBy("business_key").agg(
    spark_max("scd_version").alias("max_version")
)

# STEP 4: Identify new records (business_key not in target)
df_new = df_incoming.alias("incoming").join(
    df_current.alias("current"),
    col("incoming.business_key") == col("current.business_key"),
    "left_anti"
)

new_count = df_new.count()
print(f"New records: {new_count}")

# STEP 5: Prepare records to insert (new + changed)
# For changed records: increment version
# For new records: version = 1

df_to_insert = df_incoming.join(
    df_max_versions,
    on="business_key",
    how="left"
).withColumn(
    "scd_version",
    when(col("max_version").isNotNull(), col("max_version") + 1).otherwise(lit(1))
).withColumn(
    "is_current",
    lit(True)
).withColumn(
    "effective_from",
    current_date()
).withColumn(
    "effective_to",
    lit(None).cast("date")
).drop("max_version")

# Filter to only insert new or changed records
if changed_count > 0:
    changed_keys = [row.business_key for row in df_changed.collect()]
    df_to_insert = df_to_insert.filter(
        col("business_key").isin(changed_keys) | 
        col("business_key").isin([row.business_key for row in df_new.collect()])
    )
else:
    df_to_insert = df_new.withColumn("scd_version", lit(1))\
        .withColumn("is_current", lit(True))\
        .withColumn("effective_from", current_date())\
        .withColumn("effective_to", lit(None).cast("date"))

insert_count = df_to_insert.count()
print(f"Records to insert: {insert_count}")

# STEP 6: Insert new versions (ALL columns preserved)
if insert_count > 0:
    # Get ALL columns from the target table to ensure schema consistency
    target_columns = spark.read.table(scd_target_table).columns
    
    # Write all columns from df_to_insert that match target schema
    df_to_insert.select(*target_columns).write\
        .mode("append")\
        .saveAsTable(scd_target_table)
    
    print(f"✓ Inserted {insert_count} new/updated records with ALL columns")
else:
    print("✓ No changes detected, no records to insert")

# STEP 7: Summary
print("\n=== SCD TYPE 2 MERGE SUMMARY ===")
final_df = spark.read.table(scd_target_table)
print(f"Total records (all versions): {final_df.count()}")
print(f"Total columns: {len(final_df.columns)}")
print(f"Current records: {final_df.filter(col('is_current') == True).count()}")
print(f"Historical records: {final_df.filter(col('is_current') == False).count()}")
print(f"Unique officers: {final_df.select('business_key').distinct().count()}")

# Show version distribution
print("\n=== VERSION DISTRIBUTION ===")
final_df.groupBy("scd_version").count().orderBy("scd_version").display()

# Show officers with multiple versions (change history)
print("\n=== OFFICERS WITH CHANGE HISTORY ===")
final_df.groupBy("business_key", "officer_name").count()\
    .filter(col("count") > 1)\
    .orderBy(col("count").desc())\
    .limit(10)\
    .display()

## SCD Type 2 Implementation for Officer History Tracking

### Overview
This notebook implements **Slowly Changing Dimension (SCD) Type 2** logic to maintain full historical tracking of officer changes over time.

### Tables
* **Target**: `company_risk_intelligence_platform.silver.silver_ch_people`
* **Business Key**: `company_number | person_number` (composite key uniquely identifying each officer)
* **Change Detection**: MD5 hash of key attributes (name, role, status, nationality, residence, dates, address)

---

### SCD Type 2 Columns
| Column | Type | Description |
|--------|------|-------------|
| `business_key` | string | Composite key: company_number\|person_number |
| `attribute_hash` | string | MD5 hash of tracked attributes for change detection |
| `is_current` | boolean | TRUE for current/active record, FALSE for historical |
| `effective_from` | date | Date when this version became effective |
| `effective_to` | date | Date when this version was superseded (NULL for current) |
| `scd_version` | integer | Version number (1, 2, 3...) |

---

### Workflow

#### **Initial Load** (Cell 9)
* Overwrites table with SCD structure including ALL 62 columns (56 original + 6 SCD)
* All records marked as `is_current=TRUE`, `scd_version=1`
* Use this for first-time setup only

#### **Incremental Updates** (Cell 11)
For subsequent data loads, run Cell 11 which:

1. **Identifies Changed Records**
   * Compares `attribute_hash` between incoming and current records
   * Detects updates to officer name, role, status, address, etc.

2. **Closes Historical Versions**
   * Sets `is_current=FALSE` for superseded records
   * Sets `effective_to=current_date()`

3. **Inserts New Versions**
   * Adds changed records with `scd_version` incremented
   * Adds new officers with `scd_version=1`
   * All inserted records have `is_current=TRUE`
   * **Preserves ALL 62 columns** in new versions

4. **Preserves Unchanged Records**
   * Officers with no changes remain untouched

#### **Query Historical Changes** (Cell 10)
* Demonstrates how to query officer change history
* Shows version distribution and officers with multiple versions

---

### Usage Example

```python
# Step 1: Load new data from bronze
df = spark.read.table("company_risk_intelligence_platform.bronze.ch_people")

# Step 2: Apply transformation cells (cells 4-7)
# ... (run cells 4 through 7)

# Step 3: Run incremental merge (Cell 12)
# This automatically handles change detection and versioning
```

---

### Query Examples

**Get current state of all officers:**
```sql
SELECT *
FROM company_risk_intelligence_platform.silver.silver_ch_people
WHERE is_current = TRUE
```

**Get full change history for a specific officer:**
```sql
SELECT 
  officer_name,
  officer_role,
  is_active,
  effective_from,
  effective_to,
  scd_version
FROM company_risk_intelligence_platform.silver.silver_ch_people
WHERE person_number = '310031980001'
ORDER BY effective_from
```

**Find officers who changed roles:**
```sql
SELECT 
  business_key,
  officer_name,
  COUNT(*) as version_count
FROM company_risk_intelligence_platform.silver.silver_ch_people
GROUP BY business_key, officer_name
HAVING COUNT(*) > 1
ORDER BY version_count DESC
```

**Point-in-time query (officer state as of specific date):**
```sql
SELECT *
FROM company_risk_intelligence_platform.silver.silver_ch_people
WHERE effective_from <= '2026-05-01'
  AND (effective_to IS NULL OR effective_to > '2026-05-01')
```

---

### Monitored Attributes
Changes to these fields trigger new SCD versions:
* Officer name, role, active status
* Nationality, country of residence
* Appointment date, resignation date
* Address (line 1, locality, postal code)

**Note**: Changes to metadata fields (processing timestamps, DQ flags, links) do NOT trigger new versions.

---

### Table Schema
* **Total columns**: 62
  * 56 original silver columns (company info, officer details, demographics, dates, address, identification, verification, links, DQ flags, metadata)
  * 6 SCD tracking columns (business_key, attribute_hash, is_current, effective_from, effective_to, scd_version)
* All columns are preserved in both initial load and incremental merges